In [ ]:
pip install ultralytics opencv-python matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
from ultralytics import YOLO
import matplotlib.pyplot as plt
from google.colab import files
import tempfile

class BatchUploadTester:
    def __init__(self, model_path):
        self.model_path = model_path
        self.model = YOLO(model_path)
        self.class_names = ['dent', 'scratch', 'crack', 'glass shatter', 'lamp broken', 'tire flat']
        self.upload_dir = "/content/uploaded_images"
        os.makedirs(self.upload_dir, exist_ok=True)

    def upload_multiple_images(self):
        """Upload multiple images at once"""
        print("📤 Upload multiple images for batch testing")
        print("You can select multiple files in the file dialog")
        print("=" * 50)

        uploaded = files.upload()

        if not uploaded:
            print("❌ No files uploaded!")
            return []

        saved_paths = []
        for filename, content in uploaded.items():
            file_path = os.path.join(self.upload_dir, filename)
            with open(file_path, 'wb') as f:
                f.write(content)
            saved_paths.append(file_path)
            print(f"✅ Uploaded: {filename}")

        return saved_paths

    def test_all_images(self, image_paths, conf_threshold=0.4):
        """Test all uploaded images"""
        print(f"\n🔍 Testing {len(image_paths)} images...")
        print("=" * 50)

        for i, image_path in enumerate(image_paths, 1):
            print(f"\n📊 Image {i}/{len(image_paths)}: {os.path.basename(image_path)}")
            self.test_single_image(image_path, conf_threshold)

    def test_single_image(self, image_path, conf_threshold=0.4):
        """Test single image and show results"""
        results = self.model.predict(
            source=image_path,
            imgsz=640,
            conf=conf_threshold,
            save=False
        )

        result = results[0]

        # Print results
        self.print_results(result)

        # Display image
        self.display_image(result, os.path.basename(image_path))

    def print_results(self, result):
        """Print detection results"""
        if len(result.boxes) > 0:
            print("🛠️ DAMAGE DETECTED:")
            for i, box in enumerate(result.boxes):
                cls_id = int(box.cls[0])
                conf = box.conf[0].item()
                class_name = self.class_names[cls_id]
                print(f"   • {class_name}: {conf:.1%} confidence")
        else:
            print("✅ No damage detected")

    def display_image(self, result, filename):
        """Display image with detection boxes"""
        result_img = result.plot()

        plt.figure(figsize=(12, 8))
        plt.imshow(result_img)
        plt.axis('off')
        plt.title(f"Damage Detection: {filename}", fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

# Usage
def main():
    MODEL_PATH = r"/content/drive/MyDrive/myDataaa/dataaaa/best.pt"

    if not os.path.exists(MODEL_PATH):
        print(f"❌ Model not found: {MODEL_PATH}")
        return

    tester = BatchUploadTester(MODEL_PATH)

    print("🤖 BATCH CAR DAMAGE DETECTION")
    print("=" * 50)

    # Upload images
    image_paths = tester.upload_multiple_images()

    if image_paths:
        # Get confidence threshold
        try:
            conf = float(input("🎯 Enter confidence threshold (0.1-0.9, default 0.4): ") or "0.4")
        except:
            conf = 0.4

        # Test all images
        tester.test_all_images(image_paths, conf)

if __name__ == "__main__":
    main()